In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import os
import sys

PROJECT_PATH = '/content/drive/MyDrive/SIMUTEH/simuteh-2330511078'

os.chdir(os.path.join(PROJECT_PATH, "notebook"))

SRC_PATH = os.path.join(PROJECT_PATH, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print("Lokasi kerja:", os.getcwd())
print("Folder src ditemukan:", os.path.exists(SRC_PATH))
print("utils_cv.py tersedia:", os.path.exists(
    os.path.join(SRC_PATH, "utils_cv.py")
))
print("praproses.py tersedia:", os.path.exists(
    os.path.join(SRC_PATH, "praproses.py")
))

Lokasi kerja: /content/drive/MyDrive/SIMUTEH/simuteh-2330511078/notebook
Folder src ditemukan: True
utils_cv.py tersedia: True
praproses.py tersedia: True


In [11]:
import pandas as pd
import cv2

from utils_cv import baca, kontras_bercak
from praproses import praproses, simpan_param

print("Semua modul berhasil diimpor.")

Semua modul berhasil diimpor.


In [12]:
import os
import pandas as pd
import cv2
from utils_cv import baca, kontras_bercak
from praproses import praproses, simpan_param

PROJECT_PATH = '/content/drive/MyDrive/SIMUTEH/simuteh-2330511078'

folder_src = os.path.join(PROJECT_PATH, "src")

print("Isi folder src:")
print(os.listdir(folder_src))

Isi folder src:
['__pycache__', 'utils_cv.py', 'buat_dataset_sintesis.py', 'praproses.py']


In [14]:
import cv2
import pandas as pd
from praproses import praproses, simpan_param
from utils_cv import baca, kontras_bercak

param = {
    "gamma": 1.0,
    "clahe_clip": 2.0,
    "penapis": "median",
    "kernel": 3,
}
simpan_param(param, "../hasil/param_praproses.json")

KANAL = 2

label = pd.read_csv("../data/sintetis/label.csv")

baris = []
for _, d in label[label.kelas != "sehat"].iterrows():
    # Baca citra dan mask
    img = baca(f"../data/sintetis/citra/{d.nama_file}")
    md = cv2.imread(f"../data/sintetis/mask_daun/{d.nama_file}", 0)
    mb = cv2.imread(f"../data/sintetis/mask_bercak/{d.nama_file}", 0)

    # Terapkan fungsi praproses
    out = praproses(img, param)

    # Hitung KB sebelum dan sesudah pada kanal terpilih
    kb_sebelum = kontras_bercak(img[:, :, KANAL], md, mb)
    kb_sesudah = kontras_bercak(out[:, :, KANAL], md, mb)

    baris.append([d.kondisi_cahaya, kb_sebelum, kb_sesudah])

# 5. Buat DataFrame dan tampilkan rata-rata KB per kondisi cahaya
evaluasi = pd.DataFrame(baris, columns=["kondisi", "KB_sebelum", "KB_sesudah"])
print(evaluasi.groupby("kondisi").mean().round(2))

                KB_sebelum  KB_sesudah
kondisi                               
bayangan              4.78        8.24
gelap                 8.83       15.35
kontras_rendah        3.95        4.97
normal                8.97       10.82
terang                6.71        6.27


**Langkah 5.3 Mengerjakan Sendiri**

Menghitung simpangan baku 20 citra sehat

In [16]:
import json
import pandas as pd
import cv2
import numpy as np

from utils_cv import baca
from praproses import praproses

# Membaca parameter yang sudah disimpan pada Langkah 5.2
with open("../hasil/param_praproses.json", "r") as f:
    param = json.load(f)

KANAL = 2

label = pd.read_csv("../data/sintetis/label.csv")

baris_sehat = []

# Memproses 20 citra kelas sehat
for _, d in label[label.kelas == "sehat"].iterrows():

    img = baca(f"../data/sintetis/citra/{d.nama_file}")

    md = cv2.imread(
        f"../data/sintetis/mask_daun/{d.nama_file}", 0
    )

    # Praproses dengan parameter yang sama seperti Langkah 5.2
    out = praproses(img, param)

    # Mengambil kanal R pada area daun saja
    kanal_sebelum = img[:, :, KANAL][md > 0]
    kanal_sesudah = out[:, :, KANAL][md > 0]

    # Menghitung simpangan baku
    std_sebelum = kanal_sebelum.std()
    std_sesudah = kanal_sesudah.std()

    baris_sehat.append([
        d.nama_file,
        d.kondisi_cahaya,
        std_sebelum,
        std_sesudah
    ])

hasil_sehat = pd.DataFrame(
    baris_sehat,
    columns=[
        "nama_file",
        "kondisi",
        "std_sebelum",
        "std_sesudah"
    ]
)

# Rata-rata berdasarkan kondisi pencahayaan
tabel_sehat = (
    hasil_sehat.groupby("kondisi")[
        ["std_sebelum", "std_sesudah"]
    ].mean().round(2)
)

display(tabel_sehat)

print("Jumlah citra diproses:", len(hasil_sehat))

,std_sebelum,std_sesudah
kondisi,,
bayangan,14.37,10.32
gelap,9.34,6.52
kontras_rendah,10.87,10.74
normal,12.78,10.77
terang,11.45,12.59


Jumlah citra diproses: 20


In [17]:
hasil_sehat.to_csv(
    "../hasil/b5_uji_bercak_palsu.csv",
    index=False
)

tabel_sehat.to_csv(
    "../hasil/b5_ringkasan_bercak_palsu.csv"
)

print("Hasil pengujian berhasil disimpan.")

Hasil pengujian berhasil disimpan.


Untuk membuat panel gambar sebelum dan sesudah di proses

In [18]:
import matplotlib.pyplot as plt

# Memilih satu citra untuk setiap kondisi
pilihan = (
    hasil_sehat.drop_duplicates(subset="kondisi")
    [["nama_file", "kondisi"]]
)

fig, axes = plt.subplots(5, 2, figsize=(10, 20))

for i, data in enumerate(pilihan.itertuples(index=False)):
    nama = data.nama_file
    kondisi = data.kondisi

    img = baca(f"../data/sintetis/citra/{nama}")
    out = praproses(img, param)

    # Konversi BGR ke RGB agar warna tampil benar
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    out_rgb = cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_title(f"{kondisi} - Sebelum ({nama})")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(out_rgb)
    axes[i, 1].set_title(f"{kondisi} - Sesudah ({nama})")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.savefig("../hasil/b5_perbandingan_citra_sehat.png",
            dpi=150, bbox_inches="tight")
plt.show()

Output hidden; open in https://colab.research.google.com to view.